# Python Inheritance

> 📘 **Python Mastery** · Module 08 — Object-Oriented Programming (OOP) · Lesson 3/5

Inheritance lets a class **reuse, extend, and specialize** another class: define the general behavior once in a parent, then describe only what makes each child different. It is how `list`, web frameworks' model classes, and every PyTorch network inherit machinery from their parents.

## 🎯 Learning Objectives

- Create subclasses and explain the "is-a" relationship between child and parent
- Extend a parent constructor correctly with `super().__init__()`
- Override inherited methods and deliberately call the parent's version
- Trace method lookup through multilevel chains and read `__mro__`
- Choose between inheritance ("is-a") and composition ("has-a")
- Extend built-in types like `list` with your own methods

## 1. Parent → Child: the "is-a" Relationship

A **child class** (subclass) automatically receives every attribute and method of its **parent** (superclass). Write the parent's name in parentheses after the child's name. The mental test is always *"is-a"*: a Student **is a** Person, so `Student` may inherit from `Person` and add student-specific powers on top.

**Syntax:**

```python
class Parent:
    ...

class Child(Parent):        # Child inherits everything Parent has
    ...add or change things...
```

**Example:** `Student` gets `__init__` and `introduce()` for free.

In [1]:
class Person:                          # PARENT (base / superclass)
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def introduce(self):
        return f"I'm {self.name}, {self.age} years old."


class Student(Person):                 # CHILD (subclass): parent goes in ()
    def enroll(self, course):          # NEW ability the parent never had
        return f"{self.name} enrolled in {course}"


s = Student("Sarah", 22)               # Student reused __init__ untouched
print(s.introduce())                   # ...and introduce() came along too
print(s.enroll("Linear Algebra"))
print(isinstance(s, Person))           # a Student IS-A Person

I'm Sarah, 22 years old.
Sarah enrolled in Linear Algebra
True


## 2. `super().__init__()`: Reusing the Parent's Setup

The moment a child defines its own `__init__`, it **replaces** the parent's entirely — so parent attributes would never be created. `super().__init__(...)` delegates part of the setup upward to the parent. Rule of thumb: call it as the first line of the child's constructor.

**Syntax:**

```python
class Child(Parent):
    def __init__(self, common_args, extra_arg):
        super().__init__(common_args)     # let Parent do ITS part first
        self.extra = extra_arg            # then add child-specific state
```

**Example:**

In [2]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age


class Student(Person):
    def __init__(self, name, age, student_id):
        super().__init__(name, age)       # Person does its part of the setup
        self.student_id = student_id      # then Student adds its own


s = Student("Rafi", 19, "CSE-2026-041")
print(s.name, s.age, s.student_id)
print(isinstance(s, Person))              # still fully a Person

Rafi 19 CSE-2026-041
True


In [3]:
# What happens if you FORGET super().__init__():
class BadStudent(Person):
    def __init__(self, student_id):
        self.student_id = student_id      # Person.__init__ never ran!

bad = BadStudent("X-001")
try:
    print(bad.name)
except AttributeError as err:
    print("AttributeError:", err)         # .name was never created

AttributeError: 'BadStudent' object has no attribute 'name'


## 3. Method Overriding

When a child redefines an inherited method (same name), its version **wins** for child instances — that is *overriding*. To **extend** rather than replace, call the parent's version with `super().method()` inside your override.

**Syntax:**

```python
class Child(Parent):
    def speak(self):                  # same name -> overrides Parent.speak
        base = super().speak()        # optional: reuse the parent answer
        return base + " (extra)"
```

**Example:**

In [4]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return "..."

    def describe(self):
        return f"{self.name} says {self.speak()}"


class Dog(Animal):
    def speak(self):                      # OVERRIDE: replaces Animal.speak
        return f"{self.name}: Woof!"


rex = Dog("Rex")
generic = Animal("Mystery")

print(rex.speak())         # Dog version wins for dogs
print(generic.speak())     # the parent version still serves plain Animals
print(rex.describe())      # parent code calls the OVERRIDDEN method -- polymorphism seed

Rex: Woof!
...
Rex says Rex: Woof!


In [5]:
class RobotDog(Dog):
    def speak(self):
        base = super().speak()            # REUSE Dog's answer instead of repeating it
        return f"[beep] {base}"


robo = RobotDog("Unit-Rex")
print(robo.speak())
print(robo.describe())                    # inherited helper benefits immediately

[beep] Unit-Rex: Woof!
Unit-Rex says [beep] Unit-Rex: Woof!


## 4. Adding New Attributes and Methods

Beyond overriding, children freely grow state and skills the parent never had. Combine both moves: `super().__init__()` for shared fields, fresh assignments for new ones.

**Example:** `Student` adds `major` and can now `study()`.

In [6]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def introduce(self):
        return f"I'm {self.name}, {self.age} years old."


class Student(Person):
    def __init__(self, name, age, major):
        super().__init__(name, age)
        self.major = major                       # NEW attribute

    def study(self, hours):                      # NEW method
        return f"{self.name} studied {hours}h of {self.major}"


s = Student("Amina", 20, "Statistics")
print(s.introduce())                             # inherited, untouched
print(s.study(3))                                # brand new
print(s.major)                                   # brand new

I'm Amina, 20 years old.
Amina studied 3h of Statistics
Statistics


## 5. Checking Lineage: `isinstance()` and `issubclass()`

Two built-ins answer relationship questions: `isinstance(obj, Cls)` asks whether an *object* belongs to a class anywhere up its family tree; `issubclass(Child, Parent)` asks about two *classes* directly.

**Syntax:**

```python
isinstance(student, Person)       # object vs class
issubclass(Student, Person)       # class vs class (one direction only!)
```

**Example:**

In [7]:
class Person:
    pass


class Student(Person):
    pass


s = Student()

print(issubclass(Student, Person))
print(issubclass(Person, Student))        # False: arrows point one way
print(isinstance(s, Student), isinstance(s, Person))
print(isinstance(Person(), Student))      # a plain Person is NOT a Student

True
False
True True
False


## 6. Inheritance Chains: Who Gets Called?

Classes stack into **multilevel chains** (`Animal → Dog → Puppy`). When you access `pip.speak`, Python searches the instance, then `Puppy`, then `Dog`, then `Animal` — and stops at the **first** hit. `Cls.__mro__` (Method Resolution Order) shows the exact search list.

**Syntax:**

```python
class A: ...
class B(A): ...
class C(B): ...          # C inherits from B, which inherits from A

C.__mro__                # the lookup order: C, B, A, object
```

**Example:** three generations, one ladder.

In [8]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return "..."

    def info(self):
        return f"{self.name}: {self.speak()}"


class Dog(Animal):
    def speak(self):
        return "Woof!"


class Puppy(Dog):
    def fetch(self):                          # exists ONLY at this level
        return f"{self.name} brings the ball!"


pip = Puppy("Pip")

print(pip.fetch())    # found ON Puppy
print(pip.speak())    # not on Puppy -> walk UP: found on Dog
print(pip.info())     # not on Puppy, not on Dog -> found on Animal
print(pip.name)       # attributes resolve along the same ladder
print([c.__name__ for c in Puppy.__mro__])    # the exact search order

Pip brings the ball!
Woof!
Pip: Woof!
Pip
['Puppy', 'Dog', 'Animal', 'object']


## 7. Hierarchical Inheritance: Siblings Sharing a Parent

Several children can inherit from the **same** parent — a *hierarchical* tree. Each sibling specializes independently while the parent holds everything they have in common.

**Example:** a tiny zoo where every resident speaks differently.

In [9]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return "..."


class Cat(Animal):
    def speak(self):
        return f"{self.name}: Meow."


class Dog(Animal):
    def speak(self):
        return f"{self.name}: Woof!"


zoo = [Cat("Manik"), Dog("Rex"), Animal("Mystery")]

for resident in zoo:
    print(resident.speak())     # siblings share the interface, not the answer

Manik: Meow.
Rex: Woof!
...


## 8. Extending Built-in Types

Built-ins like `list` and `dict` are perfectly ordinary classes — legitimate parents. Subclass them to add domain-flavored methods while keeping every native behavior intact.

**Syntax:**

```python
class ScoreList(list):
    def average(self):
        return sum(self) / len(self)
```

**Example:**

In [10]:
class ScoreList(list):
    """A list of exam scores that knows its own statistics."""

    def average(self):
        return sum(self) / len(self)

    def top(self):
        return max(self) if self else None


scores = ScoreList([88, 92, 79])

scores.append(95)                     # every native list method still works
scores.sort()
print(scores)
print(scores.average(), scores.top())
print(isinstance(scores, list))       # a ScoreList IS-A list

[79, 88, 92, 95]
88.5 95
True


## 9. Multiple Inheritance and the MRO (One Screen)

Python allows a class to list **several parents**: `class Duck(Swimmer, Flyer)`. Attribute lookup follows the Method Resolution Order — a single linearized order Python computes from your class graph, visible via `__mro__`. With a *diamond* (two paths converging on one ancestor), MRO visits each ancestor exactly once, left-to-right — but this is where multiple inheritance gets subtle, so keep it rare and shallow.

**Syntax:**

```python
class Duck(Swimmer, Flyer):    # left parent wins ties
    ...

Duck.__mro__                   # Duck -> Swimmer -> Flyer -> object
```

**Example:** order decides who answers.

In [11]:
class Swimmer:
    def move(self):
        return "swimming"


class Flyer:
    def move(self):
        return "flying"


class Duck(Swimmer, Flyer):           # both parents
    pass


d = Duck()
print(d.move())                       # Swimmer comes first in the MRO
print([c.__name__ for c in Duck.__mro__])

swimming
['Duck', 'Swimmer', 'Flyer', 'object']


In [12]:
# The diamond: B and C BOTH inherit from A
class A:
    def hello(self): return "A"

class B(A):
    def hello(self): return "B"

class C(A):
    def hello(self): return "C"

class D(B, C):                        # D sees A twice through two paths
    pass


print(D().hello())                            # B wins: MRO order
print([c.__name__ for c in D.__mro__])        # D, B, C, A -- each visited once

B
['D', 'B', 'C', 'A', 'object']


## 10. Composition: "has-a" Often Beats "is-a"

Ask the relationship question before reaching for inheritance. A Car **has-an** Engine — it is not an engine — so the car should *contain* one, not descend from one. Composition means storing an object as an attribute and delegating work to it. Prefer inheritance only for genuine is-a relationships; reach for composition when you merely want to reuse capability, when the parent changes often, or when chains grow deeper than ~3 levels.

**Syntax:**

```python
class Engine:
    ...

class Car:
    def __init__(self):
        self.engine = Engine()      # Car HAS-A Engine (composition)
```

**Example:**

In [13]:
class Engine:
    def start(self):
        return "Vroom! Engine running."

    def rev(self, times):
        return "Vr" + "oo" * times + "m!"


class Car:
    def __init__(self, model, engine):
        self.model = model
        self.engine = engine             # HAS-A: stored, delegated to

    def start(self):
        return f"{self.model}: {self.engine.start()}"


car = Car("BMW M3", Engine())
print(car.start())

# Swap engines without touching Car -- impossible with rigid inheritance:
class ElectricEngine:
    def start(self):
        return "...silent hum..."
    def rev(self, times):
        return "wh" + "ii" * times + "r!"

print(Car("Tesla Model 3", ElectricEngine()).start())

BMW M3: Vroom! Engine running.
Tesla Model 3: ...silent hum...


## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Defining `__init__` in the child without calling `super().__init__()` | Parent attributes never exist; mysterious `AttributeError` later | Call `super().__init__(...)` as the first line |
| Overriding a method with a different signature (`speak(self, volume)`) | Old callers break loudly or silently | Keep parameters compatible; extend using defaults |
| Inheriting purely "to grab some functions" | Fragile coupling to a parent you don't conceptually extend | Compose, or import the helper instead |
| Reading a parent's `__private` attr from the child | Name mangling renames it per class (`_Parent__x`) | Use single `_underscore` names for protected internals |
| Guessing the winner in a diamond | The wrong class's method silently answers | Print `Cls.__mro__`; keep hierarchies shallow |

## 💡 Best Practices & Pro Tips

- Default to composition; inherit only when the is-a sentence is honestly true.
- Keep hierarchies under ~3 levels — deep trees are where bugs hide.
- Always call `super()` (never hardcode `Parent.method(self)`); it cooperates with multiple inheritance later.
- Design children so they can stand in wherever the parent is expected (the Liskov intuition) — next lesson turns that into polymorphism.
- 🤖 **AI-engineering relevance:** subclassing is the official extension API of ML stacks — `class Net(nn.Module)` (PyTorch), `class MyScaler(BaseEstimator, TransformerMixin)` (scikit-learn), custom Keras layers. Knowing exactly where `super().__init__()` goes is required literacy for writing models.

## 📌 Summary

| Syntax | Meaning | Example |
|---|---|---|
| `class Child(Parent):` | Child inherits all of Parent | `class Student(Person):` |
| `super().__init__(...)` | Run the parent's setup first | `super().__init__(name, age)` |
| redefine a method | Overriding — child version wins | `def speak(self): ...` |
| `super().method()` | Call the overridden parent version | `super().speak()` |
| `isinstance(o, P)` / `issubclass(C, P)` | Lineage tests | `issubclass(Student, Person)` |
| `Cls.__mro__` | Attribute search order | `[c.__name__ for c in Puppy.__mro__]` |
| `self.part = Part()` | Composition ("has-a") | `Car(model, Engine())` |

**Key takeaways**
- Inheritance expresses is-a: children reuse the parent wholesale and specialize.
- Own `__init__` in a child? You owe the parent a `super().__init__()` call.
- Method lookup walks the MRO upward and stops at the first match.
- When unsure between "is-a" and "has-a", choose composition — it swaps parts without rewriting the whole.

> 🔗 **Next Lesson:** [04 · Python Polymorphism](../04_Polymorphism/) — why one method name, called identically on many different objects, is the quiet superpower of OOP.